# Automotive multi-echelon supply chain — work notebook

**Proposal:** [Brainstorming_Template.ipynb](Brainstorming_Template.ipynb)  
**Rubric:** [selfDefinedProject.md](selfDefinedProject.md) — **Option 2** (2 deeper Cypher + 1 GDS)

**Dataset:** Moetz et al. (2020), Mendeley Data [DOI 10.17632/pr3sdy5vp3.1](https://doi.org/10.17632/pr3sdy5vp3.1)

**Phase 1 (this notebook scaffold):** paths, Neo4j smoke test, `.xlsb` → `data_export/*.csv`, quick pandas profiles, Mermaid model, placeholders for ingest, EDA (8+), deeper questions, GDS.

Run notebooks with **working directory** `finalProject` so relative paths resolve.

## 1. Paths and imports

In [4]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

# Project root = cwd (start Jupyter from finalProject)
ROOT = Path.cwd().resolve()
WORKBOOK = ROOT / "2020_dataset_OfAutomotiveProductionNetwork.xlsb"
EXPORT_DIR = ROOT / "data_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT", ROOT)
print("WORKBOOK exists:", WORKBOOK.is_file(), WORKBOOK)
print("EXPORT_DIR:", EXPORT_DIR)

ROOT C:\Users\b_nko\Downloads\Supply Chain
WORKBOOK exists: True C:\Users\b_nko\Downloads\Supply Chain\2020_dataset_OfAutomotiveProductionNetwork.xlsb
EXPORT_DIR: C:\Users\b_nko\Downloads\Supply Chain\data_export


## 2. Neo4j driver

Loads [`.env`](.env) via `python-dotenv`. Use the same variables as Phase 0 (`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`).

In [16]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase, basic_auth

load_dotenv(ROOT / ".env")
uri = os.environ.get("NEO4J_URI", "bolt://localhost:7687").strip()
user = os.environ.get("NEO4J_USER", "neo4j").strip()
password = os.environ.get("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE", "").strip() or None

driver = GraphDatabase.driver(uri, auth=basic_auth(user, password), database=NEO4J_DATABASE)
driver.verify_connectivity()
print("Connected:", uri, "| database:", NEO4J_DATABASE or "(default)")


def run_query(query, parameters=None):
    """Run Cypher and return a pandas DataFrame (same style as recommender notebook)."""
    parameters = parameters or {}
    sess_kw = {"database": NEO4J_DATABASE} if NEO4J_DATABASE else {}
    with driver.session(**sess_kw) as session:
        result = session.run(query, parameters)
        return pd.DataFrame([dict(record) for record in result])


run_query("RETURN 1 AS ok")

Connected: bolt://localhost:7687 | database: (default)


,ok
0,1


## 3. Extract workbook tabs → CSV (`pyxlsb`)

Expected sheets (from proposal): `products`, `nodes`, `nodes_inflow`, `arcs`, `capacity_at_arc`, `max_flow_product_per_arc`, `max_flow_group_per_arc`, `operations`, `BOM`, `demands`, `initial_inventories`, `initial_flows`.

**Neo4j `LOAD CSV`:** copy needed files into the database **import** directory; use `file:///filename.csv` in Cypher.

In [6]:
EXPECTED_SHEETS = [
    "products",
    "nodes",
    "nodes_inflow",
    "arcs",
    "capacity_at_arc",
    "max_flow_product_per_arc",
    "max_flow_group_per_arc",
    "operations",
    "BOM",
    "demands",
    "initial_inventories",
    "initial_flows",
]

if not WORKBOOK.is_file():
    print(
        "MISSING workbook. Download from Mendeley and save as:\n ",
        WORKBOOK.name,
    )
else:
    xl = pd.ExcelFile(WORKBOOK, engine="pyxlsb")
    available = set(xl.sheet_names)
    print("Sheets in file:", len(available))
    exported = []
    missing = []
    for name in EXPECTED_SHEETS:
        if name not in available:
            # try case-insensitive
            lower = {s.lower(): s for s in available}
            key = name.lower()
            if key in lower:
                name = lower[key]
            else:
                missing.append(name)
                continue
        df = pd.read_excel(WORKBOOK, sheet_name=name, engine="pyxlsb")
        out = EXPORT_DIR / f"{name}.csv"
        df.to_csv(out, index=False, encoding="utf-8")
        exported.append((name, len(df), len(df.columns)))
    for row in exported:
        print(f"  {row[0]}: {row[1]} rows × {row[2]} cols -> {row[0]}.csv")
    if missing:
        print("NOT FOUND (check exact sheet names in workbook):", missing)

Sheets in file: 12
  products: 28049 rows × 3 cols -> products.csv
  nodes: 12 rows × 1 cols -> nodes.csv
  nodes_inflow: 44 rows × 2 cols -> nodes_inflow.csv
  arcs: 11 rows × 4 cols -> arcs.csv
  capacity_at_arc: 154 rows × 4 cols -> capacity_at_arc.csv
  max_flow_product_per_arc: 365 rows × 5 cols -> max_flow_product_per_arc.csv
  max_flow_group_per_arc: 20 rows × 5 cols -> max_flow_group_per_arc.csv
  operations: 15 rows × 7 cols -> operations.csv
  BOM: 87059 rows × 3 cols -> BOM.csv
  demands: 28000 rows × 4 cols -> demands.csv
  initial_inventories: 82 rows × 6 cols -> initial_inventories.csv
  initial_flows: 117 rows × 5 cols -> initial_flows.csv


## 4. Profile exported CSVs

In [7]:
csvs = sorted(EXPORT_DIR.glob("*.csv"))
if not csvs:
    print("No CSVs in", EXPORT_DIR, "— run extraction cell above.")
else:
    for p in csvs:
        df = pd.read_csv(p, nrows=5000)
        print(f"\n=== {p.name} === shape(sample) {df.shape}")
        print(df.dtypes.head(12))
        print(df.head(3))


=== arcs.csv === shape(sample) (11, 4)
starting_node_i           object
ending_node_j             object
process_lead_time_l_ij     int64
group_g                   object
dtype: object
       starting_node_i ending_node_j  process_lead_time_l_ij group_g
0                  zp7           zp8                       0     car
1    gear-supplier_inv           zp7                       2    gear
2  engine-supplier_inv           zp7                       1  engine

=== BOM.csv === shape(sample) (5000, 3)
mother                             int64
child                             object
individual_input_quantity_q_mc     int64
dtype: object
   mother child  individual_input_quantity_q_mc
0   64001   DK8                               1
1   64002   D83                               1
2   64003   DK8                               1

=== capacity_at_arc.csv === shape(sample) (154, 4)
starting_node_i    object
ending_node_j      object
period_t            int64
capacity_c_ijt      int64
dtype: objec

## 5. Phase 2 — Tabular validation and data dictionary

**Goal:** Full row counts, nulls, cardinality, and cross-table checks before Phase 3 `LOAD CSV`.

**Modeling note:** `products.csv` is mostly numeric `product_p`, with a small set of alphanumeric SKUs (e.g. engine codes). `BOM.csv` uses the same string id space for `mother` and `child`. In Neo4j, `Product.productId` is a **trimmed string** for both `MERGE` endpoints. After Phase 3.3, `Product.catalogSource` is `'products_sheet'` or `'BOM_only'` (first appearance outside the sheet), and each `REQUIRES` edge has `source = 'BOM'`.

Run the **next two code cells**, then keep the dictionary table below aligned with your actual outputs. After Phase 3 ingest, use the **Graph vocabulary** subsection (below the table) when writing Cypher and GDS projections.

In [8]:
# Full-file profile: row counts, nulls, uniques (fits in memory for this dataset)
csvs = sorted(EXPORT_DIR.glob("*.csv"))
if not csvs:
    print("No CSVs in", EXPORT_DIR, "— run Section 3 extraction first.")
else:
    summaries = []
    for p in csvs:
        df = pd.read_csv(p, low_memory=False)
        na = df.isna().sum()
        na_nonzero = na[na > 0]
        print(f"\n{'=' * 70}\n{p.name}  rows={len(df):,}  cols={df.shape[1]}")
        print(df.dtypes.to_string())
        if len(na_nonzero):
            print("Null counts (nonzero):\n", na_nonzero.to_string())
        else:
            print("Null counts: none")
        print("Distinct counts:")
        for col in df.columns:
            print(f"  {col}: {df[col].nunique():,}")
        if "period_t" in df.columns:
            print(
                f"  period_t range: {df['period_t'].min()} … {df['period_t'].max()}"
            )
        summaries.append((p.name, len(df), list(df.columns)))
    print(f"\nTotal CSV files profiled: {len(summaries)}")


arcs.csv  rows=11  cols=4
starting_node_i           object
ending_node_j             object
process_lead_time_l_ij     int64
group_g                   object
Null counts: none
Distinct counts:
  starting_node_i: 11
  ending_node_j: 8
  process_lead_time_l_ij: 5
  group_g: 7

BOM.csv  rows=87,059  cols=3
mother                            object
child                             object
individual_input_quantity_q_mc     int64
Null counts: none
Distinct counts:
  mother: 28,049
  child: 49
  individual_input_quantity_q_mc: 4

capacity_at_arc.csv  rows=154  cols=4
starting_node_i    object
ending_node_j      object
period_t            int64
capacity_c_ijt      int64
Null counts: none
Distinct counts:
  starting_node_i: 11
  ending_node_j: 8
  period_t: 14
  capacity_c_ijt: 6
  period_t range: 61 … 74

demands.csv  rows=28,000  cols=4
node_n          object
product_p        int64
demand_d_npt     int64
period_t         int64
Null counts: none
Distinct counts:
  node_n: 1
  product_p: 28,00

In [9]:
# Cross-table checks: node ids and product ids vs master lists
paths = {p.stem: p for p in EXPORT_DIR.glob("*.csv")}


def _load(stem: str) -> pd.DataFrame:
    return pd.read_csv(paths[stem], low_memory=False)


def _numeric_ids(s: pd.Series) -> set[int]:
    v = pd.to_numeric(s, errors="coerce").dropna()
    return set(v.astype(int))


issues: list[str] = []

if "nodes" not in paths:
    issues.append("SKIP: nodes.csv missing")
else:
    nodes_df = _load("nodes")
    node_set = set(nodes_df["node_n"].dropna().astype(str))

    def nodes_ref_ok(stem: str, cols: list[str]) -> None:
        if stem not in paths:
            return
        df = _load(stem)
        for col in cols:
            if col not in df.columns:
                continue
            vals = set(df[col].dropna().astype(str))
            unknown = vals - node_set
            if unknown:
                sample = sorted(unknown)[:8]
                issues.append(
                    f"WARN {stem}.{col}: {len(unknown)} values not in nodes.node_n e.g. {sample}"
                )

    nodes_ref_ok("arcs", ["starting_node_i", "ending_node_j"])
    nodes_ref_ok("demands", ["node_n"])
    nodes_ref_ok("initial_inventories", ["node_n"])
    nodes_ref_ok("operations", ["node_n"])
    nodes_ref_ok("nodes_inflow", ["node_n"])
    for stem in (
        "capacity_at_arc",
        "max_flow_group_per_arc",
        "max_flow_product_per_arc",
        "initial_flows",
    ):
        nodes_ref_ok(stem, ["starting_node_i", "ending_node_j"])

if "products" not in paths:
    issues.append("SKIP: products.csv missing")
elif "BOM" in paths:
    products_df = _load("products")
    bom_df = _load("BOM")
    prod_int = _numeric_ids(products_df["product_p"])
    mothers = _numeric_ids(bom_df["mother"])
    missing_mothers = mothers - prod_int
    if missing_mothers:
        issues.append(
            f"WARN BOM.mother: {len(missing_mothers)} mothers not in products.product_p e.g. {sorted(missing_mothers)[:8]}"
        )
    prod_str = {str(int(x)) for x in prod_int}
    children = set(bom_df["child"].dropna().astype(str))
    overlap = children & prod_str
    issues.append(
        f"INFO BOM.child: {len(children)} distinct child codes; {len(overlap)} appear as stringified products.product_p (rest are part codes only in BOM)"
    )

if "demands" in paths and "products" in paths:
    ddf = _load("demands")
    pdf = _load("products")
    pset = _numeric_ids(pdf["product_p"])
    dprods = _numeric_ids(ddf["product_p"])
    missing_d = dprods - pset
    if missing_d:
        issues.append(
            f"WARN demands.product_p: {len(missing_d)} not in products e.g. {sorted(missing_d)[:8]}"
        )

if "initial_inventories" in paths and "products" in paths:
    inv = _load("initial_inventories")
    pdf = _load("products")
    prod_str = {str(x) for x in _numeric_ids(pdf["product_p"])}
    inv_p = set(inv["product_p"].dropna().astype(str))
    overlap_inv = inv_p & prod_str
    issues.append(
        f"INFO initial_inventories.product_p: {len(inv_p)} distinct symbols; {len(overlap_inv)} match stringified numeric products (others e.g. BEV-style codes — still model as :Product)"
    )

if not issues:
    print("PASS: no referential issues reported (or nothing to check).")
else:
    for line in issues:
        print(line)

INFO BOM.child: 49 distinct child codes; 0 appear as stringified products.product_p (rest are part codes only in BOM)
INFO initial_inventories.product_p: 45 distinct symbols; 0 match stringified numeric products (others e.g. BEV-style codes — still model as :Product)


### Column → graph mapping (fill / adjust after running Phase 2 cells)

| CSV file | Column | Neo4j target |
|----------|--------|----------------|
| `nodes.csv` | `node_n` | `(:Node {nodeId})` + `OEM` / `SupplierSite` / `Production` / `Inventory`, `siteKind`, `displayName` (Phase 3.4) |
| `products.csv` | `product_p` | `(:Product {productId})` — store as string |
| `products.csv` | `group_g` | `(:ProductGroup {groupName})` via `BELONGS_TO` |
| `products.csv` | `transportation_size_s` | `(:Product)` property |
| `products.csv` | ingest | `(:Product).catalogSource = 'products_sheet'` (Phase 3.3) |
| `BOM.csv` | `mother`, `child`, `individual_input_quantity_q_mc` | `(:Product)-[:REQUIRES {quantity, source:'BOM'}]->(:Product)`; `catalogSource` on endpoints coalesced to `'BOM_only'` if missing |
| `arcs.csv` | `starting_node_i`, `ending_node_j`, `process_lead_time_l_ij`, `group_g` | `(:Node)-[:SHIPS_TO {leadTime, productGroup, arcKey}]->(:Node)` |
| `nodes_inflow.csv` | `node_n`, `product_p` | `(:Node)-[:PRODUCES]->(:Product)` |
| `operations.csv` | `node_n`, groups, quantities, `alpha_nxy`, `beta_nxy` | `(:Node)-[:OUTPUTS_GROUP]` / `[:USES_INPUT]` (Phase 3.4) |
| `demands.csv` | `node_n`, `product_p`, `demand_d_npt`, `period_t` | `(:Customer)-[:ORDERS]->(:DemandFact)`; `HAS_DEMAND` / `FOR_PRODUCT` / `IN_PERIOD` / `AT_NODE` (Phase 3.4) |
| `initial_inventories.csv` | `node_n`, `product_p`, inventory fields, `period_t` | `(:Node)-[:HOLDS {…periodId}]->(:Product)` |
| `capacity_at_arc.csv` | arc ids + `period_t` + `capacity_c_ijt` | `CAPACITY_AT` with `capacity`, `periodId`, **`arcKey`** (Phase 3.5) |
| `max_flow_*` / `initial_flows` | arc/product/group + `period_t` + flow | `PLANNED_FLOW_TO` / `GROUP_FLOW_TO` / `INITIAL_FLOW_TO` with **`arcKey`** (Phase 3.5) |

**Periods:** derive `(:Period {periodId})` from distinct `period_t` values, or store `period_t` only on relationships — match your proposal in [Brainstorming_Template.ipynb](Brainstorming_Template.ipynb).

**Next:** Read **Graph vocabulary, `arcKey`, and time semantics** (subsection immediately below) for BOM vs capstone naming, `SHIPS_TO` / `arcKey`, time modeling, and the demand hub before running Phase 3 ingest.

### Graph vocabulary, `arcKey`, and time semantics

**BOM — `REQUIRES` vs `CONTAINS`:** A BOM row (mother → child) is modeled as `(mother:Product)-[:REQUIRES {quantity, source:'BOM'}]->(child:Product)`. Other notebooks may use **`CONTAINS`** for the same parent→child direction; only the **relationship type name** differs.

**Logistics — `SHIPS_TO`:** Inter-facility arcs use **`SHIPS_TO`** with `leadTime`, `productGroup`, and **`arcKey = fromNodeId + '|' + toNodeId`**. Sample Cypher elsewhere may say **`FLOWS_TO`** or **`SUPPLIES`** — in **this** notebook, query **`SHIPS_TO`**.

**Parallel arc relationships:** `CAPACITY_AT`, `PLANNED_FLOW_TO`, `GROUP_FLOW_TO`, and `INITIAL_FLOW_TO` reuse the same endpoints as `SHIPS_TO` and carry the same **`arcKey`**, plus `periodId` (and `productId` / `productGroup` where relevant). **Do not** copy `leadTime` onto those rels; join to `SHIPS_TO` on `arcKey` when you need lead time with capacity.

**Time:** Use **`periodId`** on time-varying **relationships** (`HOLDS`, capacity, flows). Use **`(:Period)`** nodes for demand via **`DemandFact`-`IN_PERIOD`-`Period`**. Opening inventory stays **`(Node)-[:HOLDS {periodId, …}]->(Product)`** only.

**Demand hub:** `(:Customer {customerId:'market'})-[:ORDERS]->(:DemandFact)` together with `HAS_DEMAND`, `FOR_PRODUCT`, `IN_PERIOD`, and `AT_NODE` for traversals.

## 6. Schema — constraints (Phase 3.2)

**Step 1 — Reset (next code cell): full wipe of this database**

- Drops **all** constraints returned by `SHOW CONSTRAINTS`.
- Runs **`MATCH (n) DETACH DELETE n`** once (removes every node and all attached relationships; no second query needed).

**Isolation:** set **`NEO4J_DATABASE`** in [`.env`](.env) to a dedicated database (e.g. `supplychain`) so you do not clear other coursework in the default `neo4j` database.

**Step 2 — Create:** run the next code cell (**Phase 3.2 step 2**) to create uniqueness on `Node.nodeId`, `Product.productId`, `ProductGroup.groupName`, `Period.periodId`, `DemandFact.demandKey`, `Customer.customerId` (singleton used when demand is linked to a market customer in Phase 3.4), then show `SHOW CONSTRAINTS` as a DataFrame.

**Prerequisite:** run **§2**. Skip Step 1 if the graph is already empty and you only need constraints.

**Drop only `supply_*` (no data loss):** run the optional **Phase 3.2 (optional) — Drop only `supply_*` constraints** code cell below to remove the six named `supply_*` constraints without deleting nodes.

In [11]:
# Phase 3.2 — Step 1: full reset (this Neo4j database only — see NEO4J_DATABASE in .env)
# Drops ALL constraints, then MATCH (n) DETACH DELETE n. Run §2 first.

def _qc_constraint(n: str) -> str:
    inner = str(n).strip().strip("`").replace("`", "``")
    return f"`{inner}`"


constraints_df = run_query(
    """
    SHOW CONSTRAINTS
    YIELD name AS constraintName
    RETURN constraintName AS name
    ORDER BY name
    """
)

n_drop = 0
if not constraints_df.empty and "name" in constraints_df.columns:
    for nm in constraints_df["name"].dropna().astype(str):
        run_query(f"DROP CONSTRAINT {_qc_constraint(nm)} IF EXISTS")
        n_drop += 1

run_query("MATCH (n) DETACH DELETE n")
print("Graph successfully cleared.")


Graph successfully cleared.


In [12]:
# Phase 3.2 (step 2) — create supply-chain uniqueness constraints


constraint_cypher = [
    "CREATE CONSTRAINT supply_node_nodeid IF NOT EXISTS FOR (n:Node) REQUIRE n.nodeId IS UNIQUE",
    "CREATE CONSTRAINT supply_product_productid IF NOT EXISTS FOR (p:Product) REQUIRE p.productId IS UNIQUE",
    "CREATE CONSTRAINT supply_productgroup_groupname IF NOT EXISTS FOR (g:ProductGroup) REQUIRE g.groupName IS UNIQUE",
    "CREATE CONSTRAINT supply_period_periodid IF NOT EXISTS FOR (t:Period) REQUIRE t.periodId IS UNIQUE",
    "CREATE CONSTRAINT supply_demandfact_demandkey IF NOT EXISTS FOR (f:DemandFact) REQUIRE f.demandKey IS UNIQUE",
    "CREATE CONSTRAINT supply_customer_customerid IF NOT EXISTS FOR (c:Customer) REQUIRE c.customerId IS UNIQUE",
]

for stmt in constraint_cypher:
    run_query(stmt)

display(
    run_query(
        """
    SHOW CONSTRAINTS
    YIELD name, type, entityType, labelsOrTypes, properties
    RETURN name, type, entityType, labelsOrTypes, properties
    ORDER BY name
    """
    )
)

,name,type,entityType,labelsOrTypes,properties
0,supply_customer_customerid,NODE_PROPERTY_UNIQUENESS,NODE,[Customer],[customerId]
1,supply_demandfact_demandkey,NODE_PROPERTY_UNIQUENESS,NODE,[DemandFact],[demandKey]
2,supply_node_nodeid,NODE_PROPERTY_UNIQUENESS,NODE,[Node],[nodeId]
3,supply_period_periodid,NODE_PROPERTY_UNIQUENESS,NODE,[Period],[periodId]
4,supply_product_productid,NODE_PROPERTY_UNIQUENESS,NODE,[Product],[productId]
5,supply_productgroup_groupname,NODE_PROPERTY_UNIQUENESS,NODE,[ProductGroup],[groupName]


## 7. Ingest — staged `LOAD CSV` (Phase 3)

### Phase 3.1 — Import path (server-side files)

`LOAD CSV ... FROM "file:///nodes.csv"` is resolved by **Neo4j on the machine running the database**, not by Jupyter. Your `data_export/` folder is only a staging area until those files live in the DB **import** directory.

1. In Neo4j Desktop: open your database → menu → **Open folder** → **Import** (or find `import` under the DBMS data directory).
2. Copy [`.env.example`](.env.example)’s `NEO4J_IMPORT_DIR` line into [`.env`](.env) and set it to that **absolute** path.
3. Run the **next code cell** to copy `data_export/*.csv` → `NEO4J_IMPORT_DIR`, or copy/symlink manually.

### Phase 3.3 — Period, products, BOM (`LOAD CSV`)

After CSVs are in the import folder, run the **Phase 3.3** code cell: `demands.csv` → distinct `(:Period)`, `products.csv` → `(:ProductGroup)` + `(:Product)` + `BELONGS_TO` and sets `Product.catalogSource = 'products_sheet'`, `BOM.csv` → `REQUIRES` with `quantity`, `source = 'BOM'`, and `catalogSource` on mother/child coalesced to `'BOM_only'` when the SKU was not already seen from `products.csv`.

Then run **Phase 3.4** (markdown + code): `nodes.csv` → `(:Node)` plus secondary labels (`OEM`, `SupplierSite`, `Production`, `Inventory`) and `siteKind` / `displayName`; `arcs.csv` → `SHIPS_TO` with `arcKey`; `nodes_inflow.csv` → **`PRODUCES`**; `demands.csv` → `Customer` + `ORDERS` + `DemandFact` + `HAS_DEMAND` / `FOR_PRODUCT` / `IN_PERIOD` / **`AT_NODE`**; `initial_inventories.csv` → `HOLDS`; `operations.csv` → **`OUTPUTS_GROUP`** + **`USES_INPUT`**.

Then run **Phase 3.5** (below): time-varying arc capacity and flows — `capacity_at_arc.csv`, `max_flow_product_per_arc.csv`, `max_flow_group_per_arc.csv`, `initial_flows.csv`.

**3.2** is §6 constraints — run before first ingest. Same `file:///…` naming for all `LOAD CSV` steps.

**Filenames on disk** must match Cypher (`demands.csv`, `products.csv`, `BOM.csv`, `nodes.csv`, `arcs.csv`, `nodes_inflow.csv`, `initial_inventories.csv`, `operations.csv`, `capacity_at_arc.csv`, `max_flow_product_per_arc.csv`, `max_flow_group_per_arc.csv`, `initial_flows.csv` — note `BOM.csv` capitalization).

In [17]:
# Phase 3.3 — LOAD CSV: Period, ProductGroup/Product/BELONGS_TO, BOM/REQUIRES
# Prerequisites: §6 constraints created; demands.csv, products.csv, BOM.csv in Neo4j import dir.
from IPython.display import display

period_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///demands.csv' AS row
WITH DISTINCT row.period_t AS pt
WHERE pt IS NOT NULL AND trim(toString(pt)) <> ''
MERGE (:Period {periodId: toInteger(pt)})
"""
run_query(period_cypher)

products_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///products.csv' AS row
WITH row,
  trim(toString(row.product_p)) AS pp,
  trim(toString(row.group_g)) AS gg
WHERE pp <> '' AND gg <> ''
MERGE (pg:ProductGroup {groupName: gg})
MERGE (pr:Product {productId: pp})
SET pr.transportationSize = coalesce(toInteger(trim(toString(row.transportation_size_s))), 0),
  pr.catalogSource = 'products_sheet'
MERGE (pr)-[:BELONGS_TO]->(pg)
"""
run_query(products_cypher)

bom_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///BOM.csv' AS row
WITH row,
  trim(toString(row.mother)) AS mraw,
  trim(toString(row.child)) AS craw
WHERE mraw <> '' AND craw <> ''
MERGE (parent:Product {productId: mraw})
MERGE (child:Product {productId: craw})
MERGE (parent)-[req:REQUIRES]->(child)
SET req.quantity = coalesce(toInteger(trim(toString(row.individual_input_quantity_q_mc))), 0),
  req.source = 'BOM',
  parent.catalogSource = coalesce(parent.catalogSource, 'BOM_only'),
  child.catalogSource = coalesce(child.catalogSource, 'BOM_only')
"""
run_query(bom_cypher)

display(run_query("MATCH (p:Period) RETURN count(p) AS periods"))
display(run_query("MATCH (pr:Product) RETURN count(pr) AS products"))
display(run_query("MATCH (pg:ProductGroup) RETURN count(pg) AS product_groups"))
display(run_query("MATCH ()-[b:BELONGS_TO]->() RETURN count(b) AS belongs_to"))
display(run_query("MATCH ()-[r:REQUIRES]->() RETURN count(r) AS requires"))
display(
    run_query(
        "MATCH (p:Product) WHERE p.catalogSource = 'products_sheet' RETURN count(p) AS products_from_sheet"
    )
)
display(
    run_query(
        "MATCH (p:Product) WHERE p.catalogSource = 'BOM_only' RETURN count(p) AS products_bom_only"
    )
)
display(run_query("MATCH ()-[r:REQUIRES]->() WHERE r.source = 'BOM' RETURN count(r) AS requires_with_bom_source"))

,periods
0,14


,products
0,28049


,product_groups
0,7


,belongs_to
0,28049


,requires
0,87059


,products_from_sheet
0,28049


,products_bom_only
0,0


,requires_with_bom_source
0,87059


### Phase 3.4 — Network nodes, arcs, facility–product, demand facts, inventory, operations

**Prerequisites:** Phase **3.3** has already run (so `Product`, `ProductGroup`, `Period` exist). Copy the CSVs listed in §7 into the Neo4j **import** directory.

| File | Graph pattern |
|------|----------------|
| `nodes.csv` | `(:Node {nodeId})` + labels `OEM` / `SupplierSite` / `Production` / `Inventory` (from id patterns); `siteKind`, `displayName` |
| `arcs.csv` | `(:Node)-[:SHIPS_TO {leadTime, productGroup, arcKey}]->(:Node)` |
| `nodes_inflow.csv` | `(:Node)-[:PRODUCES]->(:Product)` |
| `demands.csv` | `(:Customer {customerId:'market'})-[:ORDERS]->(:DemandFact)`; `(:Node)-[:HAS_DEMAND]->(f)-[:FOR_PRODUCT]->(:Product)`; `(f)-[:IN_PERIOD]->(:Period)`; `(f)-[:AT_NODE]->(:Node)` |
| `initial_inventories.csv` | `(:Node)-[:HOLDS {periodId, initialInventory, safetyStock, maxInventory}]->(:Product)` |
| `operations.csv` | `(:Node)-[:OUTPUTS_GROUP {inputGroup, …}]->(:ProductGroup)` (output); `(:Node)-[:USES_INPUT {inputGroup}]->(:ProductGroup)` (input) |

`HOLDS` keeps `periodId` on the relationship so the same node–product pair can appear across periods; demand time is modeled via `IN_PERIOD` to `(:Period)`. Run the next code cell after 3.3.

In [18]:
# Phase 3.4 — LOAD CSV: Node (+ domain labels), SHIPS_TO (+arcKey), PRODUCES, DemandFact + Customer,
# HOLDS, OUTPUTS_GROUP + USES_INPUT
# Prerequisites: Phase 3.3 complete; nodes.csv, arcs.csv, nodes_inflow.csv, demands.csv,
# initial_inventories.csv, operations.csv in Neo4j import dir.
from IPython.display import display

nodes_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH trim(toString(row.node_n)) AS nn
WHERE nn <> ''
MERGE (:Node {nodeId: nn})
"""
run_query(nodes_cypher)

_node_label_queries = [
    """MATCH (n:Node)
SET n.displayName = n.nodeId,
  n.siteKind = CASE WHEN n.nodeId IN ['zp7', 'zp8'] THEN 'OEM' ELSE 'supplier' END""",
    "MATCH (n:Node) WHERE n.nodeId IN ['zp7', 'zp8'] SET n:OEM",
    "MATCH (n:Node) WHERE NOT n.nodeId IN ['zp7', 'zp8'] SET n:SupplierSite",
    "MATCH (n:Node) WHERE n.nodeId ENDS WITH '_prod' SET n:Production",
    "MATCH (n:Node) WHERE n.nodeId ENDS WITH '_inv' SET n:Inventory",
]
for _q in _node_label_queries:
    run_query(_q)

arcs_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///arcs.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  row
WHERE si <> '' AND ej <> ''
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[s:SHIPS_TO]->(b)
SET s.leadTime = coalesce(toInteger(trim(toString(row.process_lead_time_l_ij))), 0),
  s.productGroup = trim(toString(row.group_g)),
  s.arcKey = si + '|' + ej
"""
run_query(arcs_cypher)

produces_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///nodes_inflow.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.product_p)) AS pid
WHERE nn <> '' AND pid <> ''
MERGE (n:Node {nodeId: nn})
MERGE (pr:Product {productId: pid})
MERGE (n)-[:PRODUCES]->(pr)
"""
run_query(produces_cypher)

demands_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///demands.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.product_p)) AS pid,
  row
WHERE nn <> '' AND pid <> ''
WITH nn, pid, row, trim(toString(row.period_t)) AS ptc
WHERE ptc <> ''
WITH nn, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (cust:Customer {customerId: 'market'})
MERGE (n:Node {nodeId: nn})
MERGE (pr:Product {productId: pid})
MERGE (t:Period {periodId: pt})
MERGE (f:DemandFact {demandKey: nn + '::' + pid + '::' + toString(pt)})
SET f.quantity = coalesce(toInteger(trim(toString(row.demand_d_npt))), 0)
MERGE (cust)-[:ORDERS]->(f)
MERGE (n)-[:HAS_DEMAND]->(f)
MERGE (f)-[:FOR_PRODUCT]->(pr)
MERGE (f)-[:IN_PERIOD]->(t)
MERGE (f)-[:AT_NODE]->(n)
"""
run_query(demands_cypher)

holds_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///initial_inventories.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.product_p)) AS pid,
  row
WHERE nn <> '' AND pid <> ''
WITH nn, pid, row, trim(toString(row.period_t)) AS ptc
WHERE ptc <> ''
WITH nn, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (n:Node {nodeId: nn})
MERGE (pr:Product {productId: pid})
MERGE (n)-[h:HOLDS {periodId: pt}]->(pr)
SET h.initialInventory = coalesce(toInteger(trim(toString(row.initial_inventory_I_np0))), 0),
  h.safetyStock = coalesce(toInteger(trim(toString(row.safety_stock))), 0),
  h.maxInventory = coalesce(toInteger(trim(toString(row.max_inventory))), 0)
"""
run_query(holds_cypher)

operations_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///operations.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.input_product_group_x)) AS gx,
  trim(toString(row.output_product_group_y)) AS gy,
  row
WHERE nn <> '' AND gx <> '' AND gy <> ''
MERGE (n:Node {nodeId: nn})
MERGE (gout:ProductGroup {groupName: gy})
MERGE (gin:ProductGroup {groupName: gx})
MERGE (n)-[out:OUTPUTS_GROUP {inputGroup: gx}]->(gout)
SET out.outputGroup = gy,
  out.inputQty = coalesce(toInteger(trim(toString(row.input_quantity_in_nxy))), 0),
  out.outputQty = coalesce(toInteger(trim(toString(row.output_quantity_out_nxy))), 0),
  out.alpha = coalesce(toFloat(trim(toString(row.alpha_nxy))), 0.0),
  out.beta = coalesce(toFloat(trim(toString(row.beta_nxy))), 0.0)
MERGE (n)-[inp:USES_INPUT {inputGroup: gx}]->(gin)
"""
run_query(operations_cypher)

display(run_query("MATCH (n:Node) RETURN count(n) AS supply_nodes"))
display(run_query("MATCH (n:OEM) RETURN count(n) AS oem_nodes"))
display(run_query("MATCH ()-[s:SHIPS_TO]->() RETURN count(s) AS ships_to"))
display(run_query("MATCH ()-[p:PRODUCES]->() RETURN count(p) AS produces"))
display(run_query("MATCH (f:DemandFact) RETURN count(f) AS demand_facts"))
display(run_query("MATCH (c:Customer) RETURN count(c) AS customers"))
display(run_query("MATCH ()-[r:HAS_DEMAND]->() RETURN count(r) AS has_demand"))
display(run_query("MATCH ()-[r:ORDERS]->() RETURN count(r) AS orders"))
display(run_query("MATCH ()-[r:AT_NODE]->() RETURN count(r) AS at_node"))
display(run_query("MATCH ()-[r:FOR_PRODUCT]->() RETURN count(r) AS for_product"))
display(run_query("MATCH ()-[r:IN_PERIOD]->() RETURN count(r) AS in_period"))
display(run_query("MATCH ()-[ho:HOLDS]->() RETURN count(ho) AS holds"))
display(run_query("MATCH ()-[o:OUTPUTS_GROUP]->() RETURN count(o) AS outputs_group"))
display(run_query("MATCH ()-[u:USES_INPUT]->() RETURN count(u) AS uses_input"))

,supply_nodes
0,12


,oem_nodes
0,2


,ships_to
0,11


,produces
0,44


,demand_facts
0,28000


...

,for_product
0,28000


,in_period
0,28000


,holds
0,82


,outputs_group
0,15


,uses_input
0,15


### Phase 3.5 — Time-varying arc capacity and flows

**Prerequisites:** Phases **3.3** and **3.4** complete (`Node`, `SHIPS_TO`, `Product`, …). Copy these CSVs into the Neo4j import directory.

Relationship types follow [Brainstorming_Template.ipynb](Brainstorming_Template.ipynb) where applicable; `CAPACITY_AT` models per-period arc capacity from `capacity_at_arc`.

| File | Pattern |
|------|---------|
| `capacity_at_arc.csv` | `(:Node)-[:CAPACITY_AT {periodId, arcKey}]->(:Node)` with property `capacity` |
| `max_flow_product_per_arc.csv` | `(:Node)-[:PLANNED_FLOW_TO {periodId, productId, arcKey}]->(:Node)` with property `plannedFlow` |
| `max_flow_group_per_arc.csv` | `(:Node)-[:GROUP_FLOW_TO {periodId, productGroup, arcKey}]->(:Node)` with property `plannedFlow` |
| `initial_flows.csv` | `(:Node)-[:INITIAL_FLOW_TO {periodId, productId, arcKey}]->(:Node)` with property `initialFlow` |

`SHIPS_TO` stays the static arc from `arcs.csv`; Phase **3.5** relationships are **parallel** to it on the same node pairs, each carrying **`arcKey`** (`from|to`, same as `SHIPS_TO`) plus `periodId` (and `productId` / `productGroup` where relevant). Join static lead time to capacity with matching `arcKey` (and period). Run the next code cell after 3.4.

In [19]:
# Phase 3.5 — LOAD CSV: CAPACITY_AT, PLANNED_FLOW_TO, GROUP_FLOW_TO, INITIAL_FLOW_TO
# Prerequisites: Phases 3.3–3.4 complete; four CSVs in Neo4j import dir.
from IPython.display import display

capacity_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///capacity_at_arc.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND ptc <> ''
WITH si, ej, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[c:CAPACITY_AT {periodId: pt}]->(b)
SET c.capacity = coalesce(toInteger(trim(toString(row.capacity_c_ijt))), 0),
  c.arcKey = si + '|' + ej
"""
run_query(capacity_cypher)

planned_product_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///max_flow_product_per_arc.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.product_p)) AS pid,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND pid <> '' AND ptc <> ''
WITH si, ej, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[f:PLANNED_FLOW_TO {periodId: pt, productId: pid}]->(b)
SET f.plannedFlow = coalesce(toInteger(trim(toString(row.planned_flow))), 0),
  f.arcKey = si + '|' + ej
"""
run_query(planned_product_cypher)

planned_group_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///max_flow_group_per_arc.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.group_g)) AS gn,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND gn <> '' AND ptc <> ''
WITH si, ej, gn, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[g:GROUP_FLOW_TO {periodId: pt, productGroup: gn}]->(b)
SET g.plannedFlow = coalesce(toInteger(trim(toString(row.planned_flow))), 0),
  g.arcKey = si + '|' + ej
"""
run_query(planned_group_cypher)

initial_flow_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///initial_flows.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.product_p)) AS pid,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND pid <> '' AND ptc <> ''
WITH si, ej, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[init:INITIAL_FLOW_TO {periodId: pt, productId: pid}]->(b)
SET init.initialFlow = coalesce(toInteger(trim(toString(row.initial_flow))), 0),
  init.arcKey = si + '|' + ej
"""
run_query(initial_flow_cypher)

display(run_query("MATCH ()-[c:CAPACITY_AT]->() RETURN count(c) AS capacity_at"))
display(run_query("MATCH ()-[f:PLANNED_FLOW_TO]->() RETURN count(f) AS planned_flow_product"))
display(run_query("MATCH ()-[g:GROUP_FLOW_TO]->() RETURN count(g) AS planned_flow_group"))
display(run_query("MATCH ()-[x:INITIAL_FLOW_TO]->() RETURN count(x) AS initial_flow_to"))
display(
    run_query(
        """
        MATCH ()-[c:CAPACITY_AT]->()
        RETURN count(*) AS capacity_rels, count(c.arcKey) AS capacity_with_arckey
        """
    )
)

,capacity_at
0,154


,planned_flow_product
0,365


,planned_flow_group
0,20


,initial_flow_to
0,117


,capacity_rels,capacity_with_arckey
0,154,154


## 8. Post-ingest checks (Phase 4)

Run **after Phases 3.3–3.5** complete (all `LOAD CSV` cells). The next code cell prints:

- **Node labels** — expect `Product` 28,049; `DemandFact` 28,000; `Period` 14; `Node` 12; `ProductGroup` 7; **`Customer` 1** (after 3.4).
- **Relationship types** — e.g. **`REQUIRES`** 87,059; **`BELONGS_TO`** 28,049; **`SHIPS_TO`** 11; **`PRODUCES`** 44; demand hub rels **28,000** each (`HAS_DEMAND`, `ORDERS`, `FOR_PRODUCT`, `IN_PERIOD`, `AT_NODE`); **`HOLDS`** 82; **`OUTPUTS_GROUP`** / **`USES_INPUT`** 15 each; **`CAPACITY_AT`** 154; **`PLANNED_FLOW_TO`** 365; **`GROUP_FLOW_TO`** 20; **`INITIAL_FLOW_TO`** 117 (row counts from the workbook CSVs).
- **`arcKey`** — `SHIPS_TO`, `CAPACITY_AT`, and parallel flow rels should have **`arcKey` populated** (`capacity_rels` = `capacity_with_arckey`, etc.).
- **Domain labels** — **`OEM`** nodes 2 (`zp7`, `zp8`).

Adjust expectations if your export differs. Use these checks after any re-ingest or constraint change.

In [20]:
from IPython.display import display

display(run_query("MATCH (n) RETURN labels(n) AS labels, count(*) AS cnt ORDER BY cnt DESC"))
display(
    run_query(
        """
        MATCH ()-[r]->()
        RETURN type(r) AS relType, count(*) AS cnt
        ORDER BY cnt DESC
        """
    )
)
display(run_query("MATCH (c:Customer) RETURN count(c) AS customers"))
display(run_query("MATCH (n:OEM) RETURN count(n) AS oem_nodes"))
display(
    run_query(
        """
        MATCH ()-[s:SHIPS_TO]->()
        RETURN count(*) AS ships_to_rels, count(s.arcKey) AS ships_to_with_arckey
        """
    )
)
display(
    run_query(
        """
        MATCH ()-[f:PLANNED_FLOW_TO]->()
        RETURN count(*) AS planned_flow_rels, count(f.arcKey) AS planned_flow_with_arckey
        """
    )
)
display(
    run_query(
        """
        MATCH ()-[x:INITIAL_FLOW_TO]->()
        RETURN count(*) AS initial_flow_rels, count(x.arcKey) AS initial_flow_with_arckey
        """
    )
)
display(
    run_query(
        """
        MATCH ()-[g:GROUP_FLOW_TO]->()
        RETURN count(*) AS group_flow_rels, count(g.arcKey) AS group_flow_with_arckey
        """
    )
)

,labels,cnt
0,[Product],28049
1,[DemandFact],28000
2,[Period],14
3,[ProductGroup],7
4,"[Node, SupplierSite, Inventory]",4
5,"[Node, SupplierSite, Production]",4
6,"[Node, OEM]",2
7,"[Node, SupplierSite]",2
8,[Customer],1


,relType,cnt
0,REQUIRES,87059
1,BELONGS_TO,28049
2,HAS_DEMAND,28000
3,ORDERS,28000
4,FOR_PRODUCT,28000
5,IN_PERIOD,28000
6,AT_NODE,28000
7,PLANNED_FLOW_TO,365
8,CAPACITY_AT,154
9,INITIAL_FLOW_TO,117


,customers
0,1


,oem_nodes
0,2


,ships_to_rels,ships_to_with_arckey
0,11,11


,planned_flow_rels,planned_flow_with_arckey
0,365,365


,initial_flow_rels,initial_flow_with_arckey
0,117,117


,group_flow_rels,group_flow_with_arckey
0,20,20


## 9. Graph EDA (Phase 5 — Supervisor Narrative)

This EDA section is organized as a decision-focused storyline:

1. **Model trust and integrity**
2. **Structure and dependency complexity**
3. **Operational pressure and bottlenecks**
4. **Decision signals and prioritization**

Each block follows: **question** -> **Cypher** -> **result table** -> **interpretation for action**.

Graph vocabulary used here matches this notebook: `Node`, `Product`, `ProductGroup`, `Period`, `DemandFact`, `REQUIRES`, `SHIPS_TO`, `CAPACITY_AT`, `PLANNED_FLOW_TO`, `GROUP_FLOW_TO`, `INITIAL_FLOW_TO`, `HOLDS`, with lane joins through `arcKey` and `periodId` where relevant.

### Section A — Model trust and integrity

### EDA 1 — Graph coverage and structural sanity check

**Question:** How are major entity layers and relationship families distributed in the loaded graph?

**Why it matters:** A compact structural baseline confirms that ingest succeeded and that downstream EDA rests on a complete graph.

**Query:** Run the code cell below to return layer counts, relationship-family counts, and role-label distributions.

**Interpretation prompts:**
- Which layers dominate the graph size?
- Do relationship families match expected workbook-derived scale?
- Are any critical labels or families unexpectedly absent?

In [21]:
from IPython.display import display

eda_1_nodes = """
MATCH (n:Node) RETURN 'Node' AS layer, count(n) AS cnt
UNION ALL MATCH (p:Product) RETURN 'Product' AS layer, count(p) AS cnt
UNION ALL MATCH (pg:ProductGroup) RETURN 'ProductGroup' AS layer, count(pg) AS cnt
UNION ALL MATCH (t:Period) RETURN 'Period' AS layer, count(t) AS cnt
UNION ALL MATCH (f:DemandFact) RETURN 'DemandFact' AS layer, count(f) AS cnt
UNION ALL MATCH (c:Customer) RETURN 'Customer' AS layer, count(c) AS cnt
"""
display(run_query(eda_1_nodes))

eda_1_rels = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, count(r) AS cnt
ORDER BY cnt DESC
"""
display(run_query(eda_1_rels))

eda_1_node_label_sets = """
MATCH (n:Node)
RETURN labels(n) AS nodeLabels, count(*) AS cnt
ORDER BY cnt DESC
"""
display(run_query(eda_1_node_label_sets))


,layer,cnt
0,Node,12
1,Product,28049
2,ProductGroup,7
3,Period,14
4,DemandFact,28000
5,Customer,1


,relationshipType,cnt
0,REQUIRES,87059
1,BELONGS_TO,28049
2,HAS_DEMAND,28000
3,ORDERS,28000
4,FOR_PRODUCT,28000
5,IN_PERIOD,28000
6,AT_NODE,28000
7,PLANNED_FLOW_TO,365
8,CAPACITY_AT,154
9,INITIAL_FLOW_TO,117


,nodeLabels,cnt
0,"[Node, SupplierSite, Inventory]",4
1,"[Node, SupplierSite, Production]",4
2,"[Node, OEM]",2
3,"[Node, SupplierSite]",2


### EDA 1 — Summary with result statistics

**Entity-layer counts**
- `Node`: **12**
- `Product`: **28,049**
- `ProductGroup`: **7**
- `Period`: **14**
- `DemandFact`: **28,000**
- `Customer`: **1**

**Relationship-family counts**
- `REQUIRES`: **87,059**
- `BELONGS_TO`: **28,049**
- Demand hub: `HAS_DEMAND`, `ORDERS`, `FOR_PRODUCT`, `IN_PERIOD`, `AT_NODE` each **28,000**
- Planning/time layer: `PLANNED_FLOW_TO` **365**, `CAPACITY_AT` **154**, `INITIAL_FLOW_TO` **117**, `GROUP_FLOW_TO` **20**
- Operations/inventory: `HOLDS` **82**, `PRODUCES` **44**, `OUTPUTS_GROUP` **15**, `USES_INPUT` **15**
- Logistics topology: `SHIPS_TO` **11**

**Node label composition**
- `[Node, SupplierSite, Inventory]`: **4**
- `[Node, SupplierSite, Production]`: **4**
- `[Node, OEM]`: **2**
- `[Node, SupplierSite]`: **2**

**Interpretation**
- **Model integrity:** Demand wiring is fully consistent because all demand-hub relationship counts match `DemandFact` (**28,000** each).
- **Structural signal:** The graph is BOM-heavy (`REQUIRES` **87,059**) with a compact facility network (`Node` **12**, `SHIPS_TO` **11**), which is expected for this dataset.
- **Analytical readiness:** Static topology, time-varying lane planning, inventory, and demand layers are all populated, so deeper bottleneck/risk analysis is well supported.

---

### Section B — Structure and dependency complexity


### EDA 2 — BOM width by product group (`REQUIRES`)

**Question:** How wide is first-hop dependency complexity for each `ProductGroup`?

**Why it matters:** Wider immediate BOM fan-out indicates higher coordination burden and potentially larger disruption blast radius.

**Query:** Run the code cell below to summarize average/max first-hop fan-out and count products with no outgoing `REQUIRES`.

**Interpretation prompts:**
- Which groups have the broadest immediate dependency footprint?
- Where do we see many products with no outgoing BOM requirements?
- Which groups should be monitored for complexity risk?

In [22]:
from IPython.display import display

eda_2_cypher = """
MATCH (pg:ProductGroup)<-[:BELONGS_TO]-(p:Product)
OPTIONAL MATCH (p)-[r:REQUIRES]->(:Product)
WITH pg, p, count(r) AS nChildren
RETURN pg.groupName AS groupName,
       count(p) AS nProducts,
       avg(toFloat(nChildren)) AS avgRequiresChildren,
       max(nChildren) AS maxRequiresChildren
ORDER BY nProducts DESC
"""
display(run_query(eda_2_cypher))

,groupName,nProducts,avgRequiresChildren,maxRequiresChildren
0,car,28000,3.10725,4
1,engine,29,1.00000,1
2,gear,8,1.00000,1
3,seat,4,2.00000,2
4,seat_componment,4,1.00000,1
5,battery_componment,3,1.00000,1
6,battery,1,4.00000,4


### Interpretation — EDA 2 (BOM width by group)

**Result highlights**
- `car` dominates both scale and width: **28,000** products, **avgRequiresChildren = 3.10725**, **maxRequiresChildren = 4**.
- Upstream groups are structurally narrower: `engine` (**29** products, avg **1**), `gear` (**8**, avg **1**), `seat` (**4**, avg **2**), `seat_componment` (**4**, avg **1**), `battery_componment` (**3**, avg **1**), `battery` (**1**, avg **4**).
- `maxRequiresChildren` is at least **1** in every group, confirming each group has at least one product with an outgoing first-hop BOM dependency.

**Why this matters**
- Dependency width is concentrated in finished cars, so downstream variant complexity is the main driver of first-hop BOM coordination risk.
- Smaller upstream groups look more linear at first hop, but can still be critical if they feed high-volume car dependencies.

**Actionable takeaway**
- Prioritize resilience and monitoring on groups/SKUs with the highest first-hop fan-out (especially `car`, and battery-related dependencies where local width is high).

### EDA 3 — Finished-car immediate dependency exposure

**Question:** Which `car` products have the largest direct component fan-out, and are BOM provenance properties consistently populated on `REQUIRES` edges?

**Why it matters:** High direct fan-out finished products are immediate candidates for fragility and procurement complexity.

**Query:** Run the code cell below to rank `car` products by direct `REQUIRES` count and verify `REQUIRES.source` population.

**Interpretation prompts:**
- Which finished products are most component-dense at tier 1?
- Are provenance fields consistently populated for auditability?
- Which SKUs deserve focused resiliency review?

In [23]:
from IPython.display import display

eda_3_cypher = """
MATCH (car:Product)-[:BELONGS_TO]->(:ProductGroup {groupName: 'car'})
OPTIONAL MATCH (car)-[rq:REQUIRES]->()
WITH car, count(rq) AS nDirectComponents
RETURN car.productId AS carProductId, nDirectComponents
ORDER BY nDirectComponents DESC
LIMIT 5
"""
display(run_query(eda_3_cypher))

eda_3_source = """
MATCH ()-[r:REQUIRES]->()
RETURN count(r) AS requires_total,
       sum(CASE WHEN r.source = 'BOM' THEN 1 ELSE 0 END) AS requires_with_bom_source
"""
display(run_query(eda_3_source))

,carProductId,nDirectComponents
0,64184,4
1,64185,4
2,64002,4
3,64099,4
4,64354,4


,requires_total,requires_with_bom_source
0,87059,87059


## Interpretation — EDA 3 (finished-car immediate dependency + BOM provenance)
Direct dependency width (top car SKUs)
The sample shows finished vehicles with four immediate child dependencies (nDirectComponents = 4 for each listed carProductId). This aligns with EDA 2’s car summary, where first-hop BOM width reaches 4 and averages about 3.1 across all car products. The “top 5” table highlights examples at the maximum observed tier‑1 fan-out; many other car SKUs may also sit at that maximum.

BOM provenance on REQUIRES
The second table reports requires_total = 87,059 and requires_with_bom_source = 87,059, meaning 100% of REQUIRES edges carry source = 'BOM'. Practically, this indicates the dependency layer is consistently attributed to the workbook BOM import, which supports traceability and reproducibility of downstream BOM-based analytics.

Why this matters
Tier‑1 fan-out is a direct driver of procurement and integration complexity at the finished-vehicle level, while full BOM sourcing tags strengthen confidence that the graph’s dependency structure matches the intended data lineage.



### EDA 4 — Finished-car BOM composition by child product group

**Question:** How is first-tier component dependency distributed across child product groups for `car` products, including BOM-only children without `BELONGS_TO`?

**Why it matters:** Component-group concentration reveals where finished-product exposure is clustered across upstream part families.

**Query:** Run the code cell below to aggregate `car -> child` `REQUIRES` edges by child product group.

**Interpretation prompts:**
- Which child groups dominate first-tier dependency mix?
- Is there meaningful dependence on BOM-only (unclassified) children?
- What does this imply about sourcing concentration?

In [24]:
from IPython.display import display

eda_4_cypher = """
MATCH (car:Product)-[:BELONGS_TO]->(:ProductGroup {groupName: 'car'})
MATCH (car)-[:REQUIRES]->(child:Product)
OPTIONAL MATCH (child)-[:BELONGS_TO]->(g:ProductGroup)
WITH coalesce(g.groupName, '(no BELONGS_TO / BOM-only)') AS childGroup, count(*) AS nRequiresEdges
RETURN childGroup, nRequiresEdges
ORDER BY nRequiresEdges DESC
"""
display(run_query(eda_4_cypher))

,childGroup,nRequiresEdges
0,engine,28000
1,gear,28000
2,seat,28000
3,battery,3003


### Interpretation — EDA 4 (finished-car tier‑1 mix by child `ProductGroup`)

**What this measures**  
Counts of **`(car)-[:REQUIRES]->(child)`** edges at **tier 1**, grouped by the child product’s `ProductGroup`.

**What we see**  
- **`engine`**, **`gear`**, and **`seat`** each have **28,000** tier‑1 edges — aligned with **28,000** `car` products, indicating a **direct tier‑1 BOM link from every finished vehicle** into each of these component families (in this extract).  
- **`battery`** has **3,003** tier‑1 edges, indicating a **subset** of finished vehicles carry a direct tier‑1 dependency into `battery` (variant/configuration structure in the BOM).

**Why it matters**  
Universal tier‑1 dependencies create **fleet-wide exposure** if those upstream families are disrupted; selective dependencies (like `battery` here) create **targeted exposure** for a smaller slice of SKUs.

**Actionable takeaway**  
Treat **`engine` / `gear` / `seat`** as **broad baseline risk** across the full car lineup; treat **`battery`** as **subset-specific** risk that should be interpreted alongside vehicle variant logic.



---

### Section C — Operational pressure and bottlenecks

### EDA 5 — Lane-first logistics profile (`SHIPS_TO`)

**Question:** Which directed lanes carry the highest logistics time cost in the sparse facility network?

**Why it matters:** In this dataset, node degree is mostly uniform, so lane-level `leadTime` differentiates schedule risk more clearly than full-node topology tables.

**Query:** Run the code cell below to list all `SHIPS_TO` lanes ranked by `leadTime`.

**Interpretation prompts:**
- Which lanes dominate lead-time risk?
- Are high lead times concentrated in specific product groups?
- Which lanes should be prioritized for timing-risk mitigation?

In [38]:
from IPython.display import display

eda_5_lane_catalog = """
MATCH (a:Node)-[s:SHIPS_TO]->(b:Node)
RETURN a.nodeId AS fromNode,
       b.nodeId AS toNode,
       s.arcKey AS arcKey,
       s.productGroup AS productGroup,
       s.leadTime AS leadTime
ORDER BY leadTime DESC, fromNode, toNode
"""
display(run_query(eda_5_lane_catalog))

,nodeId,labels,outShipsTo,inShipsTo,minLeadOut,maxLeadOut,avgLeadOut
0,zp7,"[Node, OEM]",1,4,0.0,0.0,0.0
1,seat-supplier_inv,"[Node, SupplierSite, Inventory]",1,1,0.0,0.0,0.0
2,seat-supplier_prod,"[Node, SupplierSite, Production]",1,1,0.0,0.0,0.0
3,battery-supplier_inv,"[Node, SupplierSite, Inventory]",1,1,1.0,1.0,1.0
4,battery-supplier_prod,"[Node, SupplierSite, Production]",1,1,1.0,1.0,1.0
5,gear-supplier_inv,"[Node, SupplierSite, Inventory]",1,1,2.0,2.0,2.0
6,engine-supplier_inv,"[Node, SupplierSite, Inventory]",1,1,1.0,1.0,1.0
7,zp8,"[Node, OEM]",0,1,NaN,NaN,NaN
8,seat-supplier_trans,"[Node, SupplierSite]",1,0,23.0,23.0,23.0
9,battery-supplier_trans,"[Node, SupplierSite]",1,0,20.0,20.0,20.0


### Interpretation prompt — EDA 5

- **What we see:** highest lead-time lanes and associated `productGroup` values.
- **Why it matters:** lane-level timing concentration is the primary logistics risk signal in a sparse network.
- **Action:** prioritize mitigation on the top slow lanes before capacity stress compounds delays.

### EDA 5 (supplement) — Period pressure + topology exceptions

**Question:** For a selected period, where do lead time and throughput pressure coincide, and which nodes are topology exceptions (`in/out degree != 1`)?

**Why it matters:** The key operational signal is lane-level pressure (capacity vs planned flow), while a short exception list captures non-uniform network roles without clutter.

**Query:** Run the code cell below to:
1. Join `SHIPS_TO` + `CAPACITY_AT` + `PLANNED_FLOW_TO` by shared endpoints for one period
2. Return only nodes with atypical in/out degree in `SHIPS_TO`

**Interpretation prompts:**
- Which lanes are most constrained in the selected period?
- Do high-pressure lanes also carry high lead times?
- Which exception nodes deserve focused operational monitoring?

In [40]:
from IPython.display import display

EDA_5_PERIOD_ID = 61

eda_5_period_pressure = f"""
MATCH (a:Node)-[s:SHIPS_TO]->(b:Node)
OPTIONAL MATCH (a)-[c:CAPACITY_AT]->(b)
WHERE c.periodId = {EDA_5_PERIOD_ID}
OPTIONAL MATCH (a)-[pf:PLANNED_FLOW_TO]->(b)
WHERE pf.periodId = {EDA_5_PERIOD_ID}
WITH s.arcKey AS arcKey,
     a.nodeId AS fromNode,
     b.nodeId AS toNode,
     s.productGroup AS productGroup,
     s.leadTime AS leadTime,
     max(c.capacity) AS capacity,
     sum(coalesce(pf.plannedFlow, 0)) AS plannedFlow
RETURN arcKey,
       fromNode,
       toNode,
       productGroup,
       leadTime,
       capacity,
       plannedFlow,
       CASE WHEN capacity IS NULL OR capacity = 0 THEN NULL
            ELSE 1.0 * plannedFlow / capacity
       END AS utilization
ORDER BY CASE WHEN utilization IS NULL THEN 1 ELSE 0 END,
         utilization DESC,
         leadTime DESC
"""
display(run_query(eda_5_period_pressure))

eda_5_topology_exceptions = """
MATCH (n:Node)
OPTIONAL MATCH (n)-[o:SHIPS_TO]->()
WITH n, count(o) AS outDeg
OPTIONAL MATCH ()-[i:SHIPS_TO]->(n)
WITH n, outDeg, count(i) AS inDeg
WHERE outDeg <> 1 OR inDeg <> 1
RETURN n.nodeId AS nodeId,
       labels(n) AS labels,
       outDeg,
       inDeg
ORDER BY (outDeg + inDeg) DESC, nodeId
"""
display(run_query(eda_5_topology_exceptions))

,arcKey,fromNode,toNode,productGroup,leadTime,capacity,plannedFlow,utilization
0,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,gear,2,2100,2000,0.952381
1,engine-supplier_prod|engine-supplier_inv,engine-supplier_prod,engine-supplier_inv,engine,1,2100,2000,0.952381
2,gear-supplier_inv|zp7,gear-supplier_inv,zp7,gear,2,2100,0,0.000000
3,battery-supplier_inv|zp7,battery-supplier_inv,zp7,battery,1,294,0,0.000000
4,battery-supplier_prod|battery-supplier_inv,battery-supplier_prod,battery-supplier_inv,battery,1,294,0,0.000000
5,engine-supplier_inv|zp7,engine-supplier_inv,zp7,engine,1,2100,0,0.000000
6,zp7|zp8,zp7,zp8,car,0,2000,0,0.000000
7,seat-supplier_inv|zp7,seat-supplier_inv,zp7,seat,0,2100,0,0.000000
8,seat-supplier_prod|seat-supplier_inv,seat-supplier_prod,seat-supplier_inv,seat,0,2100,0,0.000000
9,seat-supplier_trans|seat-supplier_prod,seat-supplier_trans,seat-supplier_prod,seat_componment,23,0,0,NaN


,nodeId,labels,outDeg,inDeg
0,zp7,"[Node, OEM]",1,4
1,battery-supplier_trans,"[Node, SupplierSite]",1,0
2,engine-supplier_prod,"[Node, SupplierSite, Production]",1,0
3,gear-supplier_prod,"[Node, SupplierSite, Production]",1,0
4,seat-supplier_trans,"[Node, SupplierSite]",1,0
5,zp8,"[Node, OEM]",0,1


### Interpretation — EDA 5 (lane pressure, planned flow coverage, topology exceptions)

**What this measures**  
This EDA combines three logistics signals for the selected period:  
1. lane-level pressure from `SHIPS_TO` + `CAPACITY_AT` + `PLANNED_FLOW_TO`  
2. utilization (`plannedFlow / capacity` when capacity > 0)  
3. topology exceptions where `inDeg` or `outDeg` differs from 1.

**What we see**
- Planned-flow pressure is concentrated on two production-to-inventory lanes:  
  - `gear-supplier_prod -> gear-supplier_inv`: `2000 / 2100 = 0.952`  
  - `engine-supplier_prod -> engine-supplier_inv`: `2000 / 2100 = 0.952`
- Most other lanes show `plannedFlow = 0` in this period, even when capacity exists.
- High lead-time transport lanes (`seat-supplier_trans -> seat-supplier_prod`, `battery-supplier_trans -> battery-supplier_prod`) appear with zero/undefined utilization because planned flow is not present there in this planned-flow table for the selected period.
- Topology exceptions confirm sparse-network roles:  
  - `zp7` is a consolidation node (`inDeg = 4`, `outDeg = 1`)  
  - `zp8` is sink-like (`inDeg = 1`, `outDeg = 0`)  
  - several supplier/transport nodes are source-like (`outDeg = 1`, `inDeg = 0`).

**Important data-scope note**  
In this dataset, `PLANNED_FLOW_TO` (from `max_flow_product_per_arc`) is defined only for the two production→inventory arcs above. Therefore, this EDA should be interpreted as **planned-flow concentration analysis**, not total network movement across all arcs.

**Why this matters**  
The main operational risk signal in this period is **targeted lane saturation**, not system-wide saturation. Planned throughput is concentrated on a narrow part of the network, so disruption on those lanes can have outsized impact.

**Actionable takeaway**  
Prioritize monitoring and contingency plans on the two ~95% utilized lanes first, and analyze other arcs with `INITIAL_FLOW_TO` / `CAPACITY_AT` to capture broader movement and latent constraints beyond product-level planned flow.

### EDA 6 — Demand hub integrity and concentration

**Question:** Which sites anchor the most `DemandFact` rows, and does the singleton `Customer` connect consistently to all demand facts?

**Why it matters:** Demand-link integrity is critical for trustworthy downstream demand and risk analysis.

**Query:** Run the code cell below to rank nodes by anchored demand facts and verify `Customer -> ORDERS -> DemandFact` coverage.

**Interpretation prompts:**
- Is demand concentrated at a small set of nodes?
- Does `ORDERS` cardinality match total `DemandFact` count?
- Are there any signs of incomplete demand wiring?

In [41]:
from IPython.display import display

eda_6_cypher = """
MATCH (n:Node)-[:HAS_DEMAND]->(f:DemandFact)
WITH n, count(f) AS nDemandFacts
RETURN n.nodeId AS nodeId, labels(n) AS labels, nDemandFacts
ORDER BY nDemandFacts DESC
"""
display(run_query(eda_6_cypher))

eda_6_hub = """
MATCH (c:Customer {customerId: 'market'})-[:ORDERS]->(f:DemandFact)
WITH count(DISTINCT c) AS customers, count(f) AS orderRels
MATCH (allf:DemandFact)
WITH customers, orderRels, count(allf) AS totalFacts
RETURN customers, orderRels, totalFacts, (orderRels = totalFacts) AS orders_cover_all_facts
"""
display(run_query(eda_6_hub))

,nodeId,labels,nDemandFacts
0,zp8,"[Node, OEM]",28000


,customers,orderRels,totalFacts,orders_cover_all_facts
0,1,28000,28000,True


### Interpretation — EDA 6 (demand hub integrity and concentration)

**What this measures**  
This EDA checks where `DemandFact` records are anchored (`HAS_DEMAND`) and whether the singleton market customer fully covers demand via `ORDERS`.

**What we see**
- All demand facts are anchored at a single node:
  - `zp8` (`[Node, OEM]`) with `nDemandFacts = 28,000`.
- Demand-hub cardinalities are fully consistent:
  - `customers = 1`
  - `orderRels = 28,000`
  - `totalFacts = 28,000`
  - `orders_cover_all_facts = True`

**Why this matters**  
The demand bridge is complete and internally consistent: every `DemandFact` is connected to the market customer and anchored at the same downstream node. This is a strong data-quality signal before deeper demand-propagation analysis.

**Analytical implication**  
Because all demand is concentrated at `zp8`, demand-side variance in this model is driven by **product/time composition**, not by **multiple demand locations**. Node-level demand concentration is therefore deterministic in this scenario.

**Actionable takeaway**  
Treat `zp8` as the sole downstream demand sink for this case study, and focus subsequent demand analytics on:
1. product mix over time (`FOR_PRODUCT` + `IN_PERIOD`)
2. upstream propagation through BOM (`REQUIRES`) and logistics (`SHIPS_TO`).

### EDA 7 — Period semantics and horizon alignment

**Question:** How are demand periods, opening-state periods, and in-transit initial-flow periods aligned in this dataset?

**Why it matters:** Correct period interpretation prevents misreading shortages, lead-time effects, and capacity/flow behavior.

**Query:** Run the code cell below to compare period ranges by data layer and quantify pre-horizon in-transit flow that arrives during the demand window.

**Interpretation prompts:**
- Is demand confined to `t=61..74` (14 periods)?
- Is opening inventory anchored at `t=60`?
- Do pre-horizon initial flows (`t<61`) contribute arrivals inside the demand horizon?

In [46]:
from IPython.display import display

eda_7_period_ranges = """
MATCH (f:DemandFact)-[:IN_PERIOD]->(t:Period)
RETURN 'DemandFact/IN_PERIOD' AS layer,
       min(t.periodId) AS minPeriod,
       max(t.periodId) AS maxPeriod,
       count(DISTINCT t.periodId) AS nPeriods,
       count(f) AS nRecords
UNION ALL
MATCH ()-[h:HOLDS]->()
RETURN 'HOLDS (opening inventory)' AS layer,
       min(h.periodId) AS minPeriod,
       max(h.periodId) AS maxPeriod,
       count(DISTINCT h.periodId) AS nPeriods,
       count(h) AS nRecords
UNION ALL
MATCH ()-[x:INITIAL_FLOW_TO]->()
RETURN 'INITIAL_FLOW_TO (start periods)' AS layer,
       min(x.periodId) AS minPeriod,
       max(x.periodId) AS maxPeriod,
       count(DISTINCT x.periodId) AS nPeriods,
       count(x) AS nRecords
UNION ALL
MATCH ()-[c:CAPACITY_AT]->()
RETURN 'CAPACITY_AT' AS layer,
       min(c.periodId) AS minPeriod,
       max(c.periodId) AS maxPeriod,
       count(DISTINCT c.periodId) AS nPeriods,
       count(c) AS nRecords
UNION ALL
MATCH ()-[p:PLANNED_FLOW_TO]->()
RETURN 'PLANNED_FLOW_TO' AS layer,
       min(p.periodId) AS minPeriod,
       max(p.periodId) AS maxPeriod,
       count(DISTINCT p.periodId) AS nPeriods,
       count(p) AS nRecords
ORDER BY layer
"""
display(run_query(eda_7_period_ranges))

eda_7_pre_horizon_arrivals = """
MATCH (a:Node)-[x:INITIAL_FLOW_TO]->(b:Node)
MATCH (a)-[s:SHIPS_TO]->(b)
WITH count(x) AS totalInitialFlowRels

MATCH (a:Node)-[x:INITIAL_FLOW_TO]->(b:Node)
MATCH (a)-[s:SHIPS_TO]->(b)
WHERE x.periodId < 61
WITH totalInitialFlowRels, count(x) AS preHorizonStarts

MATCH (a:Node)-[x:INITIAL_FLOW_TO]->(b:Node)
MATCH (a)-[s:SHIPS_TO]->(b)
WHERE x.periodId < 61
  AND (x.periodId + s.leadTime) >= 61
  AND (x.periodId + s.leadTime) <= 74
WITH totalInitialFlowRels,
     preHorizonStarts,
     count(x) AS preHorizonArriveInDemandWindow,
     sum(x.initialFlow) AS qtyPreHorizonArriveInDemandWindow

RETURN totalInitialFlowRels,
       preHorizonStarts,
       preHorizonArriveInDemandWindow,
       qtyPreHorizonArriveInDemandWindow
"""
display(run_query(eda_7_pre_horizon_arrivals))

,layer,minPeriod,maxPeriod,nPeriods,nRecords
0,DemandFact/IN_PERIOD,61,74,14,28000
1,HOLDS (opening inventory),60,60,1,82
2,INITIAL_FLOW_TO (start periods),39,60,5,117
3,CAPACITY_AT,61,74,14,154
4,PLANNED_FLOW_TO,61,70,10,365


,totalInitialFlowRels,preHorizonStarts,preHorizonArriveInDemandWindow,qtyPreHorizonArriveInDemandWindow
0,117,117,106,67739


### Interpretation — EDA 7 (period semantics and horizon alignment)

**What this measures**  
This EDA verifies how period ranges differ across demand, opening state, planning layers, and in-transit initialization.

**What we see**
- **Demand horizon is exactly 14 periods:** `DemandFact/IN_PERIOD = 61..74` (`nPeriods = 14`, `nRecords = 28,000`).
- **Opening inventory is a single snapshot at `t=60`:** `HOLDS = 60..60` (`nRecords = 82`).
- **Initial in-transit flows start before the demand horizon:** `INITIAL_FLOW_TO = 39..60` (`nRecords = 117`).
- **Capacity coverage spans the full demand horizon:** `CAPACITY_AT = 61..74` (`nPeriods = 14`).
- **Product-level planned flow is shorter:** `PLANNED_FLOW_TO = 61..70` (`nPeriods = 10`, `nRecords = 365`).

**Pipeline contribution to demand window**
- `totalInitialFlowRels = 117`
- `preHorizonStarts = 117`
- `preHorizonArriveInDemandWindow = 106`
- `qtyPreHorizonArriveInDemandWindow = 67,739`

This shows most initial-flow relationships that started before `t=61` are expected to arrive during the demand window, so pre-horizon periods are operationally relevant initialization state.

**Why this matters**
The model timeline is not just “14 demand days”; it is:
1. pre-horizon in-transit pipeline (`39..60`),
2. opening state at `60`,
3. demand/capacity horizon (`61..74`),
4. planned-flow coverage subset (`61..70`).

Interpreting shortages and bottlenecks without this timeline alignment would be misleading.

**Actionable takeaway**
Treat periods `<61` as required pipeline initialization (not extra demand), and interpret `PLANNED_FLOW_TO` analyses with the explicit note that product-level planned-flow coverage ends at `t=70`.

### EDA 8 — Capacity, opening inventory, and initial in-transit pressure

**Question:** Where do we observe early-period pressure from tight lane capacity, opening inventory concentration, and initial in-transit flow?

**Why it matters:** These three signals establish starting operational risk before full period dynamics are considered.

**Query:** Run the code cell below for three views: lane capacity tightness at `periodId = 61`, largest node opening inventories, and largest `INITIAL_FLOW_TO` lanes.

**Interpretation prompts:**
- Which lanes start with the tightest capacity?
- Is opening inventory concentrated or distributed across nodes?
- Which in-transit lanes may dominate early fulfillment behavior?

In [47]:
from IPython.display import display

EDA_8_PERIOD_ID = 61

eda_8_capacity = f"""
MATCH (a:Node)-[s:SHIPS_TO]->(b:Node)
MATCH (a)-[c:CAPACITY_AT]->(b)
WHERE c.periodId = {EDA_8_PERIOD_ID} AND s.arcKey = c.arcKey
RETURN s.arcKey AS arcKey,
       a.nodeId AS fromNode,
       b.nodeId AS toNode,
       s.leadTime AS leadTime,
       s.productGroup AS shipsProductGroup,
       c.capacity AS capacity,
       c.periodId AS periodId
ORDER BY c.capacity ASC
LIMIT 20
"""
display(run_query(eda_8_capacity))

eda_8_holds = """
MATCH (n:Node)-[h:HOLDS]->(:Product)
WITH n, sum(h.initialInventory) AS totalInitialInv, count(h) AS holdRels
RETURN n.nodeId AS nodeId, labels(n) AS labels, totalInitialInv, holdRels
ORDER BY totalInitialInv DESC
"""
display(run_query(eda_8_holds))

eda_8_init_flow = """
MATCH (a:Node)-[x:INITIAL_FLOW_TO]->(b:Node)
WITH a.nodeId AS fromNode, b.nodeId AS toNode, x.arcKey AS arcKey, sum(x.initialFlow) AS sumInitialFlow
RETURN fromNode, toNode, arcKey, sumInitialFlow
ORDER BY sumInitialFlow DESC
LIMIT 20
"""
display(run_query(eda_8_init_flow))

,arcKey,fromNode,toNode,leadTime,shipsProductGroup,capacity,periodId
0,seat-supplier_trans|seat-supplier_prod,seat-supplier_trans,seat-supplier_prod,23,seat_componment,0,61
1,battery-supplier_trans|battery-supplier_prod,battery-supplier_trans,battery-supplier_prod,20,battery_componment,0,61
2,battery-supplier_inv|zp7,battery-supplier_inv,zp7,1,battery,294,61
3,battery-supplier_prod|battery-supplier_inv,battery-supplier_prod,battery-supplier_inv,1,battery,294,61
4,zp7|zp8,zp7,zp8,0,car,2000,61
5,seat-supplier_inv|zp7,seat-supplier_inv,zp7,0,seat,2100,61
6,seat-supplier_prod|seat-supplier_inv,seat-supplier_prod,seat-supplier_inv,0,seat,2100,61
7,gear-supplier_inv|zp7,gear-supplier_inv,zp7,2,gear,2100,61
8,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,2,gear,2100,61
9,engine-supplier_inv|zp7,engine-supplier_inv,zp7,1,engine,2100,61


,nodeId,labels,totalInitialInv,holdRels
0,zp7,"[Node, OEM]",6052,38
1,seat-supplier_prod,"[Node, SupplierSite, Production]",6000,4
2,engine-supplier_inv,"[Node, SupplierSite, Inventory]",6000,29
3,gear-supplier_inv,"[Node, SupplierSite, Inventory]",4000,8
4,battery-supplier_prod,"[Node, SupplierSite, Production]",1860,3


,fromNode,toNode,arcKey,sumInitialFlow
0,seat-supplier_trans,seat-supplier_prod,seat-supplier_trans|seat-supplier_prod,56000
1,battery-supplier_trans,battery-supplier_prod,battery-supplier_trans|battery-supplier_prod,35860
2,gear-supplier_inv,zp7,gear-supplier_inv|zp7,4000
3,gear-supplier_prod,gear-supplier_inv,gear-supplier_prod|gear-supplier_inv,4000
4,engine-supplier_inv,zp7,engine-supplier_inv|zp7,2000
5,engine-supplier_prod,engine-supplier_inv,engine-supplier_prod|engine-supplier_inv,2000
6,battery-supplier_inv,zp7,battery-supplier_inv|zp7,200
7,battery-supplier_prod,battery-supplier_inv,battery-supplier_prod|battery-supplier_inv,159


### EDA 9 — Demand concentration by node and product group

**Question:** Where is demand concentrated across sites and product groups, and what share of total demand does each top segment represent?

**Why it matters:** Concentration identifies where service risk and planning sensitivity are highest.

**Query:** Run the code cell below to compute total and percentage demand share by node and by product group.

**Interpretation prompts:**
- How concentrated is demand at the top nodes/groups?
- Are there clear single points of demand pressure?
- Which segments should receive proactive mitigation?

In [22]:
from IPython.display import display

eda_9_nodes = """
MATCH (n:Node)-[:HAS_DEMAND]->(f:DemandFact)
WITH n, sum(f.quantity) AS totalDemandQty
WITH collect({nodeId:n.nodeId, qty:totalDemandQty}) AS rows, sum(totalDemandQty) AS grandTotal
UNWIND rows AS r
RETURN r.nodeId AS nodeId,
       r.qty AS totalDemandQty,
       round(100.0 * r.qty / grandTotal, 2) AS pctOfTotalDemand
ORDER BY totalDemandQty DESC
LIMIT 15
"""
display(run_query(eda_9_nodes))

eda_9_groups = """
MATCH (:Node)-[:HAS_DEMAND]->(f:DemandFact)-[:FOR_PRODUCT]->(p:Product)-[:BELONGS_TO]->(pg:ProductGroup)
WITH pg.groupName AS productGroup, sum(f.quantity) AS totalDemandQty
WITH collect({g:productGroup, q:totalDemandQty}) AS rows, sum(totalDemandQty) AS grandTotal
UNWIND rows AS r
RETURN r.g AS productGroup,
       r.q AS totalDemandQty,
       round(100.0 * r.q / grandTotal, 2) AS pctOfTotalDemand
ORDER BY totalDemandQty DESC
"""
display(run_query(eda_9_groups))

,nodeId,totalDemandQty,pctOfTotalDemand
0,zp8,28000,100.0


,productGroup,totalDemandQty,pctOfTotalDemand
0,car,28000,100.0


### EDA 10 — Lane utilization hotspots and repeated pressure

**Question:** Which lanes run closest to capacity over time, and which experience repeated high-utilization periods?

**Why it matters:** Persistent high utilization is a strong bottleneck precursor and resilience warning signal.

**Query:** Run the code cell below to compute lane-period utilization and repeated-pressure counts from `PLANNED_FLOW_TO` and `CAPACITY_AT`.

**Interpretation prompts:**
- Which lanes show the highest peak utilization?
- Which lanes are repeatedly stressed across periods?
- What rerouting or capacity actions are implied?

In [23]:
from IPython.display import display

eda_10_utilization = """
MATCH (a:Node)-[pf:PLANNED_FLOW_TO]->(b:Node)
MATCH (a)-[c:CAPACITY_AT]->(b)
WHERE pf.arcKey = c.arcKey
  AND pf.periodId = c.periodId
  AND c.capacity > 0
WITH a, b, pf.periodId AS periodId, pf.arcKey AS arcKey,
     sum(pf.plannedFlow) AS totalPlannedFlow,
     max(c.capacity) AS capacity
RETURN arcKey,
       a.nodeId AS fromNode,
       b.nodeId AS toNode,
       periodId,
       totalPlannedFlow,
       capacity,
       round(1.0 * totalPlannedFlow / capacity, 4) AS utilizationRatio
ORDER BY utilizationRatio DESC, totalPlannedFlow DESC
LIMIT 30
"""
display(run_query(eda_10_utilization))

eda_10_repeated_pressure = """
MATCH (a:Node)-[pf:PLANNED_FLOW_TO]->(b:Node)
MATCH (a)-[c:CAPACITY_AT]->(b)
WHERE pf.arcKey = c.arcKey
  AND pf.periodId = c.periodId
  AND c.capacity > 0
WITH pf.arcKey AS arcKey,
     pf.periodId AS periodId,
     (1.0 * sum(pf.plannedFlow) / max(c.capacity)) AS utilization
WITH arcKey,
     collect(utilization) AS utilSeries,
     sum(CASE WHEN utilization >= 0.8 THEN 1 ELSE 0 END) AS periodsAtOrAbove80,
     max(utilization) AS maxUtilization,
     avg(utilization) AS avgUtilization
RETURN arcKey,
       periodsAtOrAbove80,
       round(maxUtilization, 4) AS maxUtilization,
       round(avgUtilization, 4) AS avgUtilization,
       size(utilSeries) AS observedPeriods
ORDER BY periodsAtOrAbove80 DESC, maxUtilization DESC
LIMIT 20
"""
display(run_query(eda_10_repeated_pressure))

,arcKey,fromNode,toNode,periodId,totalPlannedFlow,capacity,utilizationRatio
0,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,61,2000,2100,0.9524
1,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,62,2000,2100,0.9524
2,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,63,2000,2100,0.9524
3,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,64,2000,2100,0.9524
4,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,65,2000,2100,0.9524
5,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,66,2000,2100,0.9524
6,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,67,2000,2100,0.9524
7,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,68,2000,2100,0.9524
8,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,69,2000,2100,0.9524
9,gear-supplier_prod|gear-supplier_inv,gear-supplier_prod,gear-supplier_inv,70,2000,2100,0.9524


,arcKey,periodsAtOrAbove80,maxUtilization,avgUtilization,observedPeriods
0,gear-supplier_prod|gear-supplier_inv,10,0.9524,0.9524,10
1,engine-supplier_prod|engine-supplier_inv,10,0.9524,0.9524,10


---

### Section D — Decision signals and prioritization

### EDA 11 — Upstream dependency-depth risk on finished products

**Question:** Which finished products depend on the deepest multi-hop BOM chains?

**Why it matters:** Deeper dependency chains increase exposure to upstream disruptions and coordination delays.

**Query:** Run the code cell below to compute bounded depth, reachable components, and dependency-path counts for finished products.

**Interpretation prompts:**
- Which finished products have the deepest dependency stacks?
- Where is breadth plus depth creating elevated exposure?
- Which SKUs should be prioritized for resilience analysis?

In [ ]:
from IPython.display import display

EDA_11_MAX_HOPS = 8

eda_11_depth = f"""
MATCH (car:Product)-[:BELONGS_TO]->(:ProductGroup {{groupName:'car'}})
MATCH p = (car)-[:REQUIRES*1..{EDA_11_MAX_HOPS}]->(comp:Product)
WITH car,
     max(length(p)) AS maxDependencyDepth,
     count(DISTINCT comp) AS nReachableComponents,
     count(p) AS nDependencyPaths
RETURN car.productId AS carProductId,
       maxDependencyDepth,
       nReachableComponents,
       nDependencyPaths
ORDER BY maxDependencyDepth DESC, nReachableComponents DESC, nDependencyPaths DESC
LIMIT 25
"""
display(run_query(eda_11_depth))

,carProductId,maxDependencyDepth,nReachableComponents,nDependencyPaths
0,64674,4,8,24
1,65014,4,8,24
2,64391,4,8,24
3,64648,4,8,24
4,64804,4,8,24
5,64935,4,8,24
6,64184,4,8,24
7,64358,4,8,24
8,64475,4,8,24
9,64611,4,8,24


### EDA 12 — Node criticality proxy for prioritization

**Question:** Which nodes combine high demand pressure, high shipping connectivity, and relatively low initial inventory?

**Why it matters:** A composite ranking helps supervisors prioritize mitigation where operational impact is likely highest.

**Query:** Run the code cell below to compute and rank:

`riskProxy = ((demandQty + 1) * (shipDegree + 1)) / (initialInventory + 1)`

**Interpretation prompts:**
- Which nodes dominate the top of the risk ranking?
- Is high risk driven more by demand, connectivity, or low inventory?
- What are the top immediate actions for the first-ranked nodes?

In [25]:
from IPython.display import display

eda_12_node_risk = """
MATCH (n:Node)
OPTIONAL MATCH (n)-[:HAS_DEMAND]->(f:DemandFact)
WITH n, coalesce(sum(f.quantity), 0) AS demandQty
OPTIONAL MATCH (n)-[:SHIPS_TO]->(outN:Node)
WITH n, demandQty, count(DISTINCT outN) AS outDegree
OPTIONAL MATCH (inN:Node)-[:SHIPS_TO]->(n)
WITH n, demandQty, outDegree, count(DISTINCT inN) AS inDegree
OPTIONAL MATCH (n)-[h:HOLDS]->(:Product)
WITH n, demandQty, outDegree, inDegree, coalesce(sum(h.initialInventory), 0) AS initialInventory
WITH n,
     demandQty,
     outDegree,
     inDegree,
     (outDegree + inDegree) AS shipDegree,
     initialInventory,
     (1.0 * (demandQty + 1) * (outDegree + inDegree + 1)) / (initialInventory + 1) AS riskProxy
RETURN n.nodeId AS nodeId,
       labels(n) AS labels,
       demandQty,
       outDegree,
       inDegree,
       shipDegree,
       initialInventory,
       round(riskProxy, 4) AS riskProxy
ORDER BY riskProxy DESC
LIMIT 20
"""
display(run_query(eda_12_node_risk))

,nodeId,labels,demandQty,outDegree,inDegree,shipDegree,initialInventory,riskProxy
0,zp8,"[Node, OEM]",28000,0,1,1,0,56002.0000
1,seat-supplier_inv,"[Node, SupplierSite, Inventory]",0,1,1,2,0,3.0000
2,battery-supplier_inv,"[Node, SupplierSite, Inventory]",0,1,1,2,0,3.0000
3,seat-supplier_trans,"[Node, SupplierSite]",0,1,0,1,0,2.0000
4,battery-supplier_trans,"[Node, SupplierSite]",0,1,0,1,0,2.0000
5,gear-supplier_prod,"[Node, SupplierSite, Production]",0,1,0,1,0,2.0000
6,engine-supplier_prod,"[Node, SupplierSite, Production]",0,1,0,1,0,2.0000
7,battery-supplier_prod,"[Node, SupplierSite, Production]",0,1,1,2,1860,0.0016
8,zp7,"[Node, OEM]",0,1,4,5,6052,0.0010
9,gear-supplier_inv,"[Node, SupplierSite, Inventory]",0,1,1,2,4000,0.0007


## 10. Deeper Cypher (Phase 6 — Option 2)

Two analytical questions with traversal explanation + commentary markdown (add below each result).

### Deep 1 — BOM / supply complexity

Which finished products have the deepest or broadest upstream chains, and what does that imply for disruption exposure?

In [ ]:
deep1_cypher = '''
// TODO: deep analysis 1
RETURN 1 AS placeholder;
'''
# run_query(deep1_cypher)
print(deep1_cypher)

### Deep 2 — Arc criticality

Which arcs look most critical when combining demand exposure, lead time, and capacity limits?

In [ ]:
deep2_cypher = '''
// TODO: deep analysis 2
RETURN 1 AS placeholder;
'''
# run_query(deep2_cypher)
print(deep2_cypher)

### GDS — analytical graph layers (read before `gds.graph.project`)

Use **separate projections** or one **narrowly defined** graph — avoid one undifferentiated mix of every relationship type.

1. **Topology / logistics:** `Node` (secondary labels `OEM`, `SupplierSite`, `Production`, `Inventory`) and **`SHIPS_TO`** — e.g. betweenness on the **11-arc** facility digraph.
2. **BOM / product:** `Product`, **`REQUIRES`**, optionally **`BELONGS_TO`** / **`ProductGroup`** — multi-hop assembly and grouping.
3. **Integrated / risk:** Combine patterns in **Cypher** first; use **`gds.graph.project`** or **`gds.graph.project.cypher`** with an explicit allow-list. Example: `SHIPS_TO` + `PRODUCES` links facilities to SKUs. Do **not** project **`REQUIRES`** together with **`CAPACITY_AT`** unless you write down what a combined edge **means** for the algorithm (direction, weights, duplicates).

**Directedness:** PageRank and betweenness assume a clear **directed** or **undirected** story; `SHIPS_TO` is **directed** (`from` → `to`). Document your choice if you reverse or undirect for an algorithm.

## 11. Graph Data Science (Phase 7)

Read **GDS — analytical graph layers** (subsection above) before choosing projection node labels and relationship types.

**Projection:** e.g. `Node` + `SHIPS_TO` for logistics-only betweenness (`OEM` / `SupplierSite` are extra labels on the same nodes — project `Node` unless you use a Cypher projection). Document **directed** vs **undirected** and any **weights** (this dataset’s `SHIPS_TO` is unweighted aside from properties you choose to map).

**Algorithm:** e.g. **betweenness** on the `SHIPS_TO` subgraph — state **time complexity** implications and interpret scores (e.g. consolidation nodes like `engine-supplier_inv`).

In [ ]:
gds_placeholder = '''
// TODO: CALL gds.graph.project(...)
// TODO: CALL gds.betweenness.stream(...) or .write(...)
RETURN 1 AS gds_placeholder;
'''
# run_query(gds_placeholder)
print(gds_placeholder)